# Vachan V2 — Per-Persona Tone Dial with Control Vectors

**What this proves:** the *generic* Hinglish spike used one fixed contrast (formal email vs WhatsApp friend). This notebook replaces those generic anchors with a **real persona's own phrases** — Hinglish samples the persona actually uses, versus clean-English translations of the same samples.

Result: a tone vector that's *tuned to that persona's voice*, not just generic Hinglish. The dial now means 'more like this specific person' vs 'more formal'.

**Why this matters for Vachan:** when the Fidelity Ring (av_cosine) shows a persona is drifting, we can dial the coeff up rather than prompt-engineer harder or burn a fine-tune.

> Runtime: ~8–12 min on a Kaggle **T4**. Run the basic spike first to confirm the environment works.

---

**Prerequisite:** complete `hinglish_control_vector_kaggle.ipynb` first (basic spike green). This notebook extends it.

## 0. Setup

1. Kaggle → **Accelerator: GPU T4** · **Internet: On**
2. No HuggingFace token needed (same non-gated mirror as the basic spike)
3. **Run All** top to bottom.

The only thing you may want to edit is Section 2 — the persona anchor phrases. We ship a worked example (a persona called *Rahul*). Swap them out for any real persona's Hinglish samples.

In [ ]:
# Same deps as the basic spike.
!pip install -q repeng transformers accelerate bitsandbytes

In [ ]:
import torch, json, textwrap
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from repeng import ControlVector, ControlModel, DatasetEntry

MODEL = "NousResearch/Meta-Llama-3.1-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token_id = tokenizer.eos_token_id

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map="auto")
model = ControlModel(model, list(range(-5, -18, -1)))
print("loaded:", MODEL)

## 1. Define Persona Anchors

This is the only section you change per persona.

**`PERSONA_NAME`** — just for labelling outputs.

**`PERSONA_HINGLISH_SAMPLES`** — real phrases this persona uses. Pull these from WhatsApp exports, CRM notes, or past transcripts. They should sound *unmistakably like that person*.

**`PERSONA_ENGLISH_TRANSLATIONS`** — clean English translations of those same ideas (not necessarily word-for-word — just the same semantic content in formal English). The vector is the *difference* between these two lists.

> **Rule of thumb:** 10–20 pairs = good vector. Under 8 = weak. More than 40 = diminishing returns.

In [ ]:
# ─── EDIT THIS BLOCK FOR YOUR PERSONA ──────────────────────────────────────

PERSONA_NAME = "Rahul"  # change to real persona name

# Positive anchors — authentic Hinglish phrases this persona uses.
# Pull from WhatsApp exports, call transcripts, CRM notes, etc.
PERSONA_HINGLISH_SAMPLES = [
    "haan bhai ho gaya, testing kal se start hogi",
    "kal tak bhejo, boss impatient ho raha hai",
    "ek kaam karo — pehle staging pe check karo",
    "sab theek hai, bas server thoda slow tha",
    "yaar itna time kyun lag raha deployment mein",
    "abhi chal raha hai, 10 min mein done ho jayega",
    "client ko bol do chill kare, kaam chal raha hai",
    "bhai jaldi karo, meeting mein batana hai",
    "okay bhai, push kar diya, review kar lo",
    "thoda fix karna tha, ab sab set hai",
    "staging se production pe move kar diya",
    "arey yaar bug tha, sort ho gaya",
    "haan send kar diya, dekh lo",
    "kal tak done, pakka",
    "bhai tension mat lo, main handle kar lunga",
]

# Negative anchors — clean English translation of the same ideas.
PERSONA_ENGLISH_TRANSLATIONS = [
    "Yes, it's done. Testing will begin tomorrow.",
    "Please send it by tomorrow. The manager is impatient.",
    "Here is what I'd suggest — verify on staging first.",
    "Everything is fine. The server was slightly slow.",
    "Why is the deployment taking this long?",
    "It is running now. It will be done in 10 minutes.",
    "Please inform the client to be patient. Work is in progress.",
    "Please expedite. We need to present this in the meeting.",
    "I have pushed the changes. Please review.",
    "There was a minor fix required. Everything is set now.",
    "I have moved the build from staging to production.",
    "There was a bug. It has been resolved.",
    "Yes, I have sent it. Please take a look.",
    "It will be done by tomorrow. I assure you.",
    "Please do not worry. I will handle it.",
]

# ─── END EDIT BLOCK ─────────────────────────────────────────────────────────

assert len(PERSONA_HINGLISH_SAMPLES) == len(PERSONA_ENGLISH_TRANSLATIONS), \
    f"Mismatch: {len(PERSONA_HINGLISH_SAMPLES)} Hinglish vs {len(PERSONA_ENGLISH_TRANSLATIONS)} English — they must be equal length"

print(f"Persona: {PERSONA_NAME}")
print(f"Anchor pairs: {len(PERSONA_HINGLISH_SAMPLES)}")
print("---")
for h, e in list(zip(PERSONA_HINGLISH_SAMPLES, PERSONA_ENGLISH_TRANSLATIONS))[:3]:
    print(f"  Hinglish : {h}")
    print(f"  English  : {e}")
    print()

## 2. Build Contrastive Dataset from Persona Anchors

For each anchor pair we create the same framed prompt — same user question, same chat template. The *system prompt* is the persona anchor phrase itself. repeng reads the model's hidden state at each suffix position and computes the difference between the Hinglish and English directions.

We still add generic suffixes so we capture the direction across many token positions (a wider sample = cleaner vector).

In [ ]:
# Generic suffixes — same as the basic spike. They just give repeng more token
# positions to read the hidden state from. Don't need to be persona-specific.
SUFFIXES = [
    "", "I", "I think", "Let me", "Sure", "Okay", "Honestly",
    "The plan", "We should", "Yeah", "Right", "So", "Alright",
]

# A shared user turn — same for positive and negative.
# This question is neutral; the persona anchor in the system prompt does the work.
USER_TURN = "Give me a quick update on the project."

def make_entry(anchor_positive: str, anchor_negative: str, suffix: str) -> DatasetEntry:
    def frame(anchor):
        msgs = [
            {"role": "system", "content": f"You are {PERSONA_NAME}. Here is a sample of how you talk: '{anchor}'"},
            {"role": "user",   "content": USER_TURN},
        ]
        base = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        return base + suffix
    return DatasetEntry(positive=frame(anchor_positive), negative=frame(anchor_negative))

dataset = [
    make_entry(h, e, s)
    for h, e in zip(PERSONA_HINGLISH_SAMPLES, PERSONA_ENGLISH_TRANSLATIONS)
    for s in SUFFIXES
]

print(f"{len(dataset)} contrastive pairs ({len(PERSONA_HINGLISH_SAMPLES)} anchors × {len(SUFFIXES)} suffixes)")
print("--- positive sample (tail) ---")
print(dataset[0].positive[-200:])

## 3. Train the Per-Persona Control Vector

Same one-pass extraction as the basic spike — no gradients. The vector here encodes *this persona's specific voice direction* rather than the generic Hinglish/English axis.

In [ ]:
model.reset()
persona_vector = ControlVector.train(model, tokenizer, dataset)

_layers = list(persona_vector.directions.keys())
print(f"trained persona vector for '{PERSONA_NAME}'")
print(f"layers: {len(_layers)}, direction shape: {persona_vector.directions[_layers[0]].shape}")

## 4. Save the Vector

This is the key difference from the basic spike — we **save the trained vector** as a `.pt` file. In production you'd load this per-persona and inject it at inference time without re-training.

Each persona gets their own file. Name pattern: `{persona_name}_tone_vector.pt`

In [ ]:
import os

SAVE_DIR = "/kaggle/working/vectors"
os.makedirs(SAVE_DIR, exist_ok=True)

vec_path = f"{SAVE_DIR}/{PERSONA_NAME.lower()}_tone_vector.pt"
torch.save(persona_vector, vec_path)
print(f"saved: {vec_path}")
print(f"size: {os.path.getsize(vec_path) / 1024:.1f} KB")

## 5. Turn the Dial — Same Prompt, Three Tone Settings

Now we test with a neutral prompt that isn't in the training anchors.

- `coeff +2` → model sounds more like this persona's Hinglish voice
- `coeff  0` → untouched model
- `coeff -2` → model sounds formal English (opposite of this persona's Hinglish style)

In [ ]:
def generate(prompt: str, coeff: float, persona_context: str = "") -> str:
    model.reset()
    if coeff != 0:
        model.set_control(persona_vector, coeff)
    sys_content = (
        f"You are {PERSONA_NAME}. {persona_context}".strip()
        if persona_context
        else f"You are {PERSONA_NAME}."
    )
    msgs = [
        {"role": "system", "content": sys_content},
        {"role": "user",   "content": prompt},
    ]
    ids = tokenizer.apply_chat_template(msgs, return_tensors="pt", add_generation_prompt=True).to(model.device)
    out = model.generate(
        ids,
        max_new_tokens=100,
        do_sample=False,
        repetition_penalty=1.3,
        pad_token_id=tokenizer.eos_token_id,
    )
    model.reset()
    return tokenizer.decode(out[0, ids.shape[-1]:], skip_special_tokens=True).strip()

# Test 1 — project update (similar to training domain)
PROMPT_1 = "Can you give me an update on the deployment?"
print(f"=== Test 1: '{PROMPT_1}' ===")
for c in [-2.0, 0.0, 2.0]:
    print(f"\ncoeff {c:+}")
    print(generate(PROMPT_1, c))

In [ ]:
# Test 2 — out-of-domain prompt: does the persona voice transfer to a new topic?
# This is the harder test — the vector should generalise beyond the training anchors.
PROMPT_2 = "We have a client call tomorrow. What should I prepare?"
print(f"=== Test 2 (OOD): '{PROMPT_2}' ===")
for c in [-2.0, 0.0, 2.0]:
    print(f"\ncoeff {c:+}")
    print(generate(PROMPT_2, c))

## 6. Fidelity Check — Cosine Similarity to Persona Anchors

This mirrors what Vachan's Fidelity Ring already does: compute the average cosine similarity between the generated text's embedding and the persona's anchor embeddings.

**av_cosine > 0.70** → persona voice is holding.
**av_cosine < 0.60** → dial needs to go up (or anchors need more samples).

We use a lightweight sentence encoder (`all-MiniLM-L6-v2`) — same family as what the Fidelity Ring uses — so the numbers are comparable.

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

emb_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

anchor_embs = emb_model.encode(PERSONA_HINGLISH_SAMPLES, normalize_embeddings=True)
anchor_centroid = anchor_embs.mean(axis=0)

def fidelity_score(text: str) -> float:
    e = emb_model.encode([text], normalize_embeddings=True)[0]
    return cosine(e, anchor_centroid)

print(f"Fidelity Ring — av_cosine to {PERSONA_NAME}'s Hinglish centroid")
print("-" * 55)
for prompt_label, prompt in [("update Q", PROMPT_1), ("client call Q", PROMPT_2)]:
    for c in [-2.0, 0.0, 2.0]:
        resp = generate(prompt, c)
        score = fidelity_score(resp)
        bar = "█" * int(score * 20)
        print(f"  coeff {c:+} | {prompt_label:14s} | av_cosine={score:.3f} {bar}")
    print()

## 7. Multi-Coeff Sweep — Find the Sweet Spot

A quick sweep across coeff values to see where the persona voice peaks before the text starts to degenerate. The sweet spot is usually the highest coeff where av_cosine keeps rising. Beyond that, text degenerates (repetition, gibberish).

This gives you the **recommended coeff** to hard-code per persona — or to use as the upper bound for an adaptive dial.

In [ ]:
print(f"Coeff sweep — prompt: '{PROMPT_1}'")
print("-" * 60)
results = []
for coeff in [-2.5, -1.5, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0]:
    resp = generate(PROMPT_1, coeff)
    score = fidelity_score(resp)
    results.append((coeff, score, resp))
    bar = "█" * int(score * 20)
    trunc = resp[:70].replace("\n", " ")
    print(f"  {coeff:+.1f} | av_cosine={score:.3f} {bar}")
    print(f"       {trunc}...")

best_coeff, best_score, best_resp = max(results, key=lambda x: x[1])
print(f"\n→ Recommended coeff for {PERSONA_NAME}: {best_coeff} (av_cosine={best_score:.3f})")
print(f"  Response: {best_resp[:120]}")

# Save the recommended coeff alongside the vector
config = {
    "persona": PERSONA_NAME,
    "recommended_coeff": best_coeff,
    "best_av_cosine": round(best_score, 4),
    "anchor_count": len(PERSONA_HINGLISH_SAMPLES),
    "layer_band": "range(-5, -18, -1)",
    "model": MODEL,
}
cfg_path = f"{SAVE_DIR}/{PERSONA_NAME.lower()}_vector_config.json"
with open(cfg_path, "w") as f:
    json.dump(config, f, indent=2)
print(f"\nconfig saved: {cfg_path}")
print(json.dumps(config, indent=2))

## 8. Results + What's Next

**What to look for:**
- `coeff +2` response should sound noticeably more like the persona's Hinglish voice than `coeff 0`
- av_cosine at `+2` should be higher than at `0` — that's the quantitative proof
- Test 2 (OOD prompt) generalises → the vector captured *voice*, not just topic

**If the effect is weak:**
- Add more anchor pairs (aim for 20–30)
- Widen the layer band: change `range(-5, -18, -1)` to `range(-3, -22, -1)`
- Make sure Hinglish samples are *distinctively* Hinglish — generic polite phrases won't give the model enough signal

**If text degenerates at coeff 2.0:**
- Lower to 1.0–1.5 — the sweet spot the sweep found is your production value

**Path into Vachan production (documented in README):**
1. **Store:** upload `{persona}_tone_vector.pt` + `{persona}_vector_config.json` to object storage (S3/GCS) — one file per persona
2. **Load at inference:** a small vLLM/transformers worker loads the vector and applies `model.set_control(vector, coeff)` before each generation
3. **Trigger:** the Fidelity Ring's av_cosine gate fires when persona drift detected → worker applies the vector with `recommended_coeff` → re-scores
4. **Adaptive dial (later):** instead of fixed `recommended_coeff`, binary-search coeff until av_cosine crosses 0.70 threshold per call

**Output files in `/kaggle/working/vectors/`:**

In [ ]:
import os
print("Files ready to download from Kaggle:")
for f in os.listdir(SAVE_DIR):
    size = os.path.getsize(os.path.join(SAVE_DIR, f)) / 1024
    print(f"  {f}  ({size:.1f} KB)")